### ЗАДАЧА: Пакетная загрузка конфигов деплоя

От DevOps-команды приходит пакет строк с конфигами сервисов для выкладки.
Нужно обработать их так, чтобы:
- валидные конфиги попали в итоговый список,
- проблемные записи не остановили весь пакет,
- по ошибкам собрался отдельный журнал,
- в конце было видно, какие сервисы включены по окружениям и какой у них средний timeout.

Часть строк содержит ошибки в формате и числах,
часть использует неизвестное окружение или неправильный флаг включения.


In [ ]:
print(122)
# service|max_retries|timeout_sec|environment|enabled
rows = [
    'auth|3|1.5|prod|on',
    'billing|0|2.0|stage|on',
    'search|two|0.8|dev|off',
    'media|5|-1|prod|on',
    'chat|4|1.2|test|off',
    'mail|2|0.5|stage|maybe',
    'worker|1|3.4|prod|on',
]


class DeployConfigError(Exception):
    pass


class RowFormatError(DeployConfigError):
    pass


class RetriesError(DeployConfigError):
    pass


class TimeoutError(DeployConfigError):
    pass


class EnvironmentError(DeployConfigError):
    pass


class EnabledFlagError(DeployConfigError):
    pass


def parse_config(row):
    service, max_retries, timeout_sec, environment, enabled = parts

    # TODO: распарсить строку и провалидировать max_retries, timeout_sec, environment, enabled
    # TODO: при ошибках конвертации использовать raise ... from ...
    parts = row.split("|")
    if len(parts) != 5:
        raise RowFormatError(f"Строка '{row}' не разделена на 5 частей символом '|'")
    
    try:
        max_retries = int(max_retries)
        if max_retries < 0:
            raise RetriesError(f"max_retries ({max_retries}) должен быть > 0  в строке '{row}'")
    except ValueError as e:
        raise RetriesError(f"max_retries ('{parts}') не является целым числом в строке '{row}'") from e

    try:
        timeout_sec = float(timeout_sec)
        if timeout_sec < 0:
            raise TimeoutError(f"timeout_sec ({timeout_sec}) должен быть > 0 в строке '{row}'")
    except ValueError as e:
        raise TimeoutError(f"timeout_sec ('{parts}') не является целым числом в строке '{row}'") from e
    
    environment = parts.lower()
    valid_envs = {"prod", "stage", "dev"}
    if environment not in valid_envs:
        raise EnvironmentError(f"environment ({environment}) не входит в {valid_envs} в строке '{row}'")

    # TODO: enabled вернуть как bool
    enabled_str = parts.lower()
    if enabled_str == "on":
        enabled = True
    elif enabled_str == "off":
        enabled = False
    else:
        raise EnabledFlagError(f"enabled ({enabled_str}) должен быть 'on' или 'off' в строке {row}")

    return {
        "service": service,
        "max_retries": max_retries,
        "timeout_sec": timeout_sec,
        "environment": environment,
        "enabled": enabled
    }
print(123)

def load_configs(rows):
    # TODO: вернуть (configs, errors)
    configs = []
    errors = []

    for row in rows:
        try:
            config = parse_config(row)
            configs.append(config)
        except DeployConfigError as e:
            errors.append(str(e))
    return configs, errors

# TODO: вызвать load_configs(rows)
configs, errors = load_configs(rows)

# TODO: вывести число валидных конфигов и число ошибок
print(f"Число валидных конфигов: {len(configs)}")
print(f"Число ошибок: {len(errors)}")

# TODO: вывести ошибки по типам
error_types = {}
for _,error_type,_ in errors:
    error_types[error_type] = error_types.get(error_type, 0) + 1
print(error_types)

# TODO: собрать enabled_by_environment: dict[str, list[str]]
enabled_by_environment = {}
for config in configs:
    if config["enabled"]:
        env = config["environment"]
        if env not in enabled_by_environment:
            enabled_by_environment[env] = []
        enabled_by_environment[env].append(config["service"])
print(enabled_by_environment)

# TODO: посчитать average_timeout только по enabled=True
enabled_configs = [config for config in configs if config["enabled"]]
if enabled_configs:
    total_timeout = sum(config["timeout_sec"] for config in enabled_configs)
    count_enabled = len(enabled_configs)
    average_timeout = total_timeout / count_enabled  
    print(f"Средний timeout: {average_timeout} секунд")
else:
    print("Конфигов не найдено")